In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

In [ ]:
arquivos_csv = list(RAW_DIR.glob('*.csv'))

def carregar_dados_csv():
    dados = {}

    arquivos = {
        'orders':'olist_orders_dataset.csv',
        'customers':'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'order_payments': 'olist_order_payments_dataset.csv',
        'order_reviews': 'olist_order_reviews_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv'
    }

    for nome, arquivo in arquivos.items():
        caminho = RAW_DIR / arquivo
        if caminho.exists():
            try:
                dados[nome] = pd.read_csv(caminho)
            except Exception as e:
                print(f"Erro ao carregar o arquivo {caminho}: {e}")
        else:
            print(f"Arquivo {caminho} não encontrado.")
    return dados

In [ ]:
dados = carregar_dados_csv()

In [ ]:
def visao_geral(df, nome):
    print(f"Visão geral do DataFrame: {nome.upper()}")
    print("-" * 50)
    print("Dimensões:", df.shape)
    print("\nTipos de dados:")
    print(df.dtypes)
    print("\nValores ausentes:")
    print(df.isnull().sum())
    print("\nEstatísticas descritivas:")
    print(df.describe(include='all'))
    print("\nValores únicos por coluna:")
    for coluna in df.columns:
        print(f"{coluna}: {df[coluna].nunique()} valores únicos")
    print("-" * 50)

In [ ]:
for nome, df in dados.items():
    visao_geral(df, nome)

In [ ]:
print("Análise de relacionamento entre tabelas")

relacionamentos = {
    'orders': {
        'pk': 'order_id',
        'fks': ['customer_id']
    },

    'customers': {
        'pk': 'customer_id',
        'fks': []
    },

    'products': {
        'pk': 'product_id',
        'fks': []
    },

    'order_items': {
        'pk': 'order_item_id',
        'fks': ['order_id', 'product_id', 'seller_id']
    },

    'order_payments': {
        'pk': None,
        'fks': ['order_id']
    },

    'order_reviews': {
        'pk': None,
        'fks': ['order_id']
    },

    'sellers': {
        'pk': 'seller_id',
        'fks': []
    }
}

for tabela, info in relacionamentos.items():
    if tabela in dados:
        print(f"Tabela: {tabela.upper()}")
        print(f"Chave primária: {info['pk']}")
        print(f"Chaves estrangeiras: {info['fks']}")
        print("-" * 50)

In [ ]:
#Resumo das tabelas

lista = []

for name, df in dados.items():
    lista.append({
        'Tabela': name,
        'Registros': len(df),  
        'Colunas': len(df.columns),
        'memoria (MB)': df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

resumo = pd.DataFrame(lista)
print(resumo.to_string(index=False))

In [ ]:
resumo.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'resumo_tabelas.csv', index=False)
print("\nResumo das tabelas salvo em 'data/processed/resumo_tabelas.csv'")